<a href="https://colab.research.google.com/github/Habibaaboalhassan66/swimming-detection/blob/main/Phase1RiskDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Mounted at /content/drive


In [3]:
"""
phase1_angle_calculation.py  v2
────────────────────────────────
PHASE 1 of 2 — Biomechanical Angle & Feature Calculation
Breaststroke + Butterfly

PURPOSE:
  Reads keypoint coordinates from both pose estimation JSONs,
  validates each keypoint, calculates biomechanical angles and
  symmetry measurements per frame, flags risky patterns, and
  saves results to biomechanical_features.csv for Phase 2.

INPUTS:
  pose_estimation_results_breaststroke.json
  pose_estimation_results_butterfly.json

OUTPUT:
  biomechanical_features.csv  (one row per frame, both strokes)
  annotated_frames/           (visualized frames per video)

FEATURES CALCULATED:
  Shoulder : width, y_difference, normalized_asymmetry,
             line_angle, midpoint x/y
  Head     : trunk_tilt_angle, head_midshoulder_dx/dy
  Elbow    : left/right elbow angle, elbow_angle_difference
  Upper arm: left/right orientation, orientation_difference
  Wrist    : vertical/horizontal difference, distance from midshoulder
  Combined : arm_symmetry_score
  Risk     : flags, number_of_flags, frame_risk_level

HOW TO USE IN COLAB:
  Cell 1:
    !pip install numpy pandas opencv-python-headless tqdm -q
  Cell 2:
    from google.colab import drive
    drive.mount("/content/drive")
  Cell 3:
    paste and run this file
"""

# ══════════════════════════════════════════════════════════════════════
#  ① CONFIG
# ══════════════════════════════════════════════════════════════════════
BASE = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters"

# Input JSONs
BREASTSTROKE_JSON = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/pose_estimation_results_breaststroke.json"
BUTTERFLY_JSON    = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/pose_estimation_results_butterfly.json"

# Output
CSV_SAVE_PATH  ="/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/biomechanical_features_new.csv"
ANNOTATED_DIR  = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/annotated_frames_new"

# Keypoint confidence threshold
MIN_KEYPOINT_CONFIDENCE = 0.70

# Minimum keypoints to process a frame
MIN_VALID_KEYPOINTS = 5

# ── Risk thresholds (configurable) ────────────────────────────────────
THRESHOLDS = {
    "normalized_shoulder_asymmetry":          0.15,
    "trunk_tilt_angle_abs":                   15.0,
    "elbow_angle_difference":                 25.0,
    "upper_arm_orientation_difference":       30.0,
    "normalized_wrist_vertical_difference":   0.20,
    "normalized_wrist_horizontal_difference": 0.25,
}

# ── Frame source folders (for visualization) ───────────────────────────
BREASTSTROKE_TEST_DIR = f"{BASE}/Dataset/Video/processed_frames_v4/breaststroke_testing"
BUTTERFLY_TEST_DIR    = f"{BASE}/Dataset/Video/processed_frames_v4/butterfly_testing (1)"

# ══════════════════════════════════════════════════════════════════════
#  ② IMPORTS
# ══════════════════════════════════════════════════════════════════════
import json
import math
import numpy as np
import pandas as pd
import cv2
from pathlib import Path
from tqdm import tqdm
from datetime import datetime

print("✅ Libraries loaded")

# ══════════════════════════════════════════════════════════════════════
#  ③ LOAD BOTH JSONs
# ══════════════════════════════════════════════════════════════════════
def load_json(path: str) -> dict:
    if not Path(path).exists():
        print(f"  ❌ Not found: {path}")
        return {}
    with open(path) as f:
        return json.load(f)


def get_frames_iter(video_data: dict) -> list:
    """
    Handle both JSON formats:
      - breaststroke: frames is a list of dicts with 'frame' key
      - butterfly:    frames is a list of dicts with 'frame' key
    Returns list of (frame_name, frame_data) tuples.
    """
    frames = video_data.get("frames", {})

    if isinstance(frames, dict):
        # Old format: {frame_name: frame_data}
        return [(k, v) for k, v in frames.items() if v is not None]
    elif isinstance(frames, list):
        # New format: [{frame: name, keypoints: {...}, ...}]
        result = []
        for fd in frames:
            if fd is None:
                continue
            name = fd.get("frame", "")
            result.append((name, fd))
        return result
    return []


def find_frame_on_disk(video_name: str, frame_name: str,
                       saved_path: str, stroke: str) -> str:
    """Find the actual frame file on disk."""
    if saved_path and Path(saved_path).exists():
        return saved_path

    # Search in stroke-specific folder
    if stroke == "breaststroke":
        candidate = Path(BREASTSTROKE_TEST_DIR) / video_name / frame_name
    else:
        candidate = Path(BUTTERFLY_TEST_DIR) / video_name / frame_name

    if candidate.exists():
        return str(candidate)

    return ""


# ══════════════════════════════════════════════════════════════════════
#  ④ KEYPOINT VALIDATION
# ══════════════════════════════════════════════════════════════════════
def validate_keypoints(keypoints: dict) -> tuple:
    """
    Validate all 7 keypoints.
    Returns (valid_kp dict, quality string, n_valid count).
    """
    all_kp_names = [
        "head", "left_shoulder", "right_shoulder",
        "left_elbow", "right_elbow",
        "left_wrist", "right_wrist"
    ]

    valid_kp = {}

    for name in all_kp_names:
        kp = keypoints.get(name)
        if kp is None:
            continue

        x = kp.get("x")
        y = kp.get("y")
        if x is None or y is None:
            continue
        try:
            if math.isnan(float(x)) or math.isnan(float(y)):
                continue
        except (TypeError, ValueError):
            continue

        conf = kp.get("confidence", 1.0)
        if conf < MIN_KEYPOINT_CONFIDENCE:
            continue

        # Skip if coordinates are None (butterfly format)
        if x is None or y is None:
            continue

        valid_kp[name] = {
            "x": float(x),
            "y": float(y),
            "confidence": float(conf)
        }

    n_valid = len(valid_kp)

    if n_valid == 7:
        quality = "valid"
    elif n_valid >= MIN_VALID_KEYPOINTS:
        quality = "partially_valid"
    else:
        quality = "invalid"

    return valid_kp, quality, n_valid


# ══════════════════════════════════════════════════════════════════════
#  ⑤ GEOMETRY HELPERS
# ══════════════════════════════════════════════════════════════════════
def calculate_angle(a: dict, b: dict, c: dict) -> float:
    """Angle at point B formed by A-B-C. Returns degrees or NaN."""
    try:
        ba = np.array([a["x"] - b["x"], a["y"] - b["y"]])
        bc = np.array([c["x"] - b["x"], c["y"] - b["y"]])
        dot      = np.dot(ba, bc)
        norm_ba  = np.linalg.norm(ba)
        norm_bc  = np.linalg.norm(bc)
        cos_ang  = np.clip(dot / (norm_ba * norm_bc + 1e-6), -1.0, 1.0)
        return float(math.degrees(math.acos(cos_ang)))
    except Exception:
        return float("nan")


def euclidean_distance(p1: dict, p2: dict) -> float:
    return math.sqrt((p1["x"] - p2["x"])**2 + (p1["y"] - p2["y"])**2)


def orientation_angle(p1: dict, p2: dict) -> float:
    """Angle of line p1→p2 relative to horizontal. Returns degrees."""
    return math.degrees(math.atan2(p2["y"] - p1["y"], p2["x"] - p1["x"]))


def normalize_angle_diff(diff: float) -> float:
    """Normalize angular difference to [0, 180]."""
    diff = abs(diff) % 360
    return 360 - diff if diff > 180 else diff


# ══════════════════════════════════════════════════════════════════════
#  ⑥ FEATURE CALCULATION
# ══════════════════════════════════════════════════════════════════════
def calculate_features(valid_kp: dict, frame_id: str,
                       frame_name: str, quality: str,
                       video_name: str, stroke: str) -> dict:
    """Calculate all biomechanical features for one frame."""
    NaN = float("nan")

    features = {
        "frame_id":          frame_id,
        "frame_name":        frame_name,
        "video_name":        video_name,
        "stroke":            stroke,
        "frame_quality":     quality,
        "n_valid_keypoints": len(valid_kp),
    }

    # Raw keypoint coordinates
    for kp_name in ["head", "left_shoulder", "right_shoulder",
                    "left_elbow", "right_elbow",
                    "left_wrist", "right_wrist"]:
        kp = valid_kp.get(kp_name)
        features[f"{kp_name}_x"] = kp["x"] if kp else NaN
        features[f"{kp_name}_y"] = kp["y"] if kp else NaN

    ls = valid_kp.get("left_shoulder")
    rs = valid_kp.get("right_shoulder")

    # ── SHOULDER FEATURES ─────────────────────────────────────────────
    if ls and rs:
        shoulder_width = euclidean_distance(ls, rs)
        shoulder_y_diff = abs(ls["y"] - rs["y"])

        features["shoulder_width"]                = shoulder_width
        features["shoulder_y_difference"]         = shoulder_y_diff
        features["normalized_shoulder_asymmetry"] = (
            shoulder_y_diff / (shoulder_width + 1e-6)
        )
        sla = orientation_angle(ls, rs)
        features["shoulder_line_angle_signed"] = sla
        features["shoulder_line_angle_abs"]    = abs(sla)

        mid_x = (ls["x"] + rs["x"]) / 2
        mid_y = (ls["y"] + rs["y"]) / 2
        features["mid_shoulder_x"] = mid_x
        features["mid_shoulder_y"] = mid_y
    else:
        for k in ["shoulder_width", "shoulder_y_difference",
                  "normalized_shoulder_asymmetry",
                  "shoulder_line_angle_signed", "shoulder_line_angle_abs",
                  "mid_shoulder_x", "mid_shoulder_y"]:
            features[k] = NaN
        shoulder_width = NaN
        mid_x = NaN
        mid_y = NaN

    # ── HEAD / TRUNK FEATURES ─────────────────────────────────────────
    hd = valid_kp.get("head")
    if hd and ls and rs:
        dx = hd["x"] - mid_x
        dy = hd["y"] - mid_y
        features["head_midshoulder_dx"]      = dx
        features["head_midshoulder_dy"]      = dy
        tilt = math.degrees(math.atan2(dx, abs(dy) + 1e-6))
        features["trunk_tilt_angle_signed"]  = tilt
        features["trunk_tilt_angle_abs"]     = abs(tilt)
    else:
        for k in ["head_midshoulder_dx", "head_midshoulder_dy",
                  "trunk_tilt_angle_signed", "trunk_tilt_angle_abs"]:
            features[k] = NaN

    # ── ELBOW ANGLE FEATURES ──────────────────────────────────────────
    le = valid_kp.get("left_elbow")
    re = valid_kp.get("right_elbow")
    lw = valid_kp.get("left_wrist")
    rw = valid_kp.get("right_wrist")

    features["left_elbow_angle"]  = calculate_angle(ls, le, lw) if (ls and le and lw) else NaN
    features["right_elbow_angle"] = calculate_angle(rs, re, rw) if (rs and re and rw) else NaN

    lea = features["left_elbow_angle"]
    rea = features["right_elbow_angle"]
    if not math.isnan(lea) and not math.isnan(rea):
        diff = abs(lea - rea)
        features["elbow_angle_difference"]            = diff
        features["normalized_elbow_angle_difference"] = diff / 180.0
    else:
        features["elbow_angle_difference"]            = NaN
        features["normalized_elbow_angle_difference"] = NaN

    # ── UPPER ARM ORIENTATION ─────────────────────────────────────────
    features["left_upper_arm_orientation"]  = orientation_angle(ls, le) if (ls and le) else NaN
    features["right_upper_arm_orientation"] = orientation_angle(rs, re) if (rs and re) else NaN

    lua = features["left_upper_arm_orientation"]
    rua = features["right_upper_arm_orientation"]
    if not math.isnan(lua) and not math.isnan(rua):
        diff = normalize_angle_diff(lua - rua)
        features["upper_arm_orientation_difference"]            = diff
        features["normalized_upper_arm_orientation_difference"] = diff / 180.0
    else:
        features["upper_arm_orientation_difference"]            = NaN
        features["normalized_upper_arm_orientation_difference"] = NaN

    # ── WRIST FEATURES ────────────────────────────────────────────────
    if lw and rw and ls and rs and not math.isnan(shoulder_width):
        sw = max(shoulder_width, 1e-6)

        wrist_v_diff = abs(lw["y"] - rw["y"])
        features["wrist_vertical_difference"]             = wrist_v_diff
        features["normalized_wrist_vertical_difference"]  = wrist_v_diff / sw

        left_dx  = lw["x"] - mid_x
        right_dx = rw["x"] - mid_x
        features["left_wrist_dx"]  = left_dx
        features["right_wrist_dx"] = right_dx

        wrist_h_diff = abs(abs(left_dx) - abs(right_dx))
        features["wrist_horizontal_difference"]            = wrist_h_diff
        features["normalized_wrist_horizontal_difference"] = wrist_h_diff / sw

        mid_pt   = {"x": mid_x, "y": mid_y}
        l_dist   = euclidean_distance(lw, mid_pt)
        r_dist   = euclidean_distance(rw, mid_pt)
        features["left_wrist_distance_from_midshoulder"]  = l_dist
        features["right_wrist_distance_from_midshoulder"] = r_dist
        dist_diff = abs(l_dist - r_dist)
        features["wrist_distance_difference"]             = dist_diff
        features["normalized_wrist_distance_difference"]  = dist_diff / sw
    else:
        for k in ["wrist_vertical_difference",
                  "normalized_wrist_vertical_difference",
                  "left_wrist_dx", "right_wrist_dx",
                  "wrist_horizontal_difference",
                  "normalized_wrist_horizontal_difference",
                  "left_wrist_distance_from_midshoulder",
                  "right_wrist_distance_from_midshoulder",
                  "wrist_distance_difference",
                  "normalized_wrist_distance_difference"]:
            features[k] = NaN

    # ── ARM SYMMETRY SCORE ────────────────────────────────────────────
    norm_vals = [
        features.get("normalized_elbow_angle_difference",            NaN),
        features.get("normalized_upper_arm_orientation_difference",  NaN),
        features.get("normalized_wrist_vertical_difference",         NaN),
        features.get("normalized_wrist_horizontal_difference",       NaN),
    ]
    valid_norms = [v for v in norm_vals if not math.isnan(v)]
    features["arm_symmetry_score"] = sum(valid_norms) if valid_norms else NaN

    return features


# ══════════════════════════════════════════════════════════════════════
#  ⑦ RISK FLAG GENERATION
# ══════════════════════════════════════════════════════════════════════
def generate_risk_flags(features: dict) -> tuple:
    """Apply rule-based risk flags. Returns (flags list, n_flags, risk_level)."""
    flags = []

    def check(key, label, note):
        val = features.get(key)
        if val is None or math.isnan(val):
            return
        thresh = THRESHOLDS.get(key)
        if thresh and val > thresh:
            flags.append(f"{label} ({key}={val:.3f} > {thresh}) — {note}")

    check("normalized_shoulder_asymmetry",
          "High shoulder asymmetry",
          "Matsuura et al. (2022)")
    check("trunk_tilt_angle_abs",
          "High trunk/head lateral tilt",
          "Heinlein & Cosgarea (2010)")
    check("elbow_angle_difference",
          "Large left-right elbow angle difference",
          "Asymmetric arm pull")
    check("upper_arm_orientation_difference",
          "Large upper-arm orientation difference",
          "Asymmetric recovery")
    check("normalized_wrist_vertical_difference",
          "Uneven wrist height",
          "Uneven arm recovery")
    check("normalized_wrist_horizontal_difference",
          "Uneven wrist horizontal extension",
          "Vasiliadis et al. (2019)")

    n = len(flags)
    level = "low" if n == 0 else ("medium" if n <= 2 else "high")
    return flags, n, level


# ══════════════════════════════════════════════════════════════════════
#  ⑧ VISUALIZATION
# ══════════════════════════════════════════════════════════════════════
def draw_on_frame(frame_path: str, valid_kp: dict,
                  features: dict, output_path: str) -> None:
    """Draw keypoints + measurements + risk level on frame."""
    frame = cv2.imread(frame_path)
    if frame is None:
        return

    CONNECTIONS = [
        ("left_shoulder",  "right_shoulder"),
        ("left_shoulder",  "left_elbow"),
        ("left_elbow",     "left_wrist"),
        ("right_shoulder", "right_elbow"),
        ("right_elbow",    "right_wrist"),
    ]
    COLORS = {
        "head":            (0,   50,  255),
        "left_shoulder":   (0,   165, 255),
        "right_shoulder":  (0,   165, 255),
        "left_elbow":      (0,   255, 255),
        "right_elbow":     (0,   255, 255),
        "left_wrist":      (0,   255, 50),
        "right_wrist":     (0,   255, 50),
    }

    h, w = frame.shape[:2]

    # Skeleton lines
    for p1n, p2n in CONNECTIONS:
        p1 = valid_kp.get(p1n)
        p2 = valid_kp.get(p2n)
        if p1 and p2:
            cv2.line(frame,
                     (int(p1["x"]), int(p1["y"])),
                     (int(p2["x"]), int(p2["y"])),
                     (0, 0, 0), 2)

    # Head to shoulder midpoint
    hd = valid_kp.get("head")
    ls = valid_kp.get("left_shoulder")
    rs = valid_kp.get("right_shoulder")
    if hd and ls and rs:
        mx = int((ls["x"] + rs["x"]) / 2)
        my = int((ls["y"] + rs["y"]) / 2)
        cv2.line(frame, (int(hd["x"]), int(hd["y"])), (mx, my), (0, 0, 0), 2)

    # Keypoints
    for name, kp in valid_kp.items():
        x, y  = int(kp["x"]), int(kp["y"])
        color = COLORS.get(name, (255, 255, 255))
        cv2.circle(frame, (x, y), 8, color, -1)
        cv2.circle(frame, (x, y), 2, (255, 255, 255), -1)

    # Info panel
    NaN = float("nan")
    rl  = features.get("frame_risk_level", "unknown")
    panel_items = [
        ("Sh asym",   features.get("normalized_shoulder_asymmetry", NaN),
         THRESHOLDS["normalized_shoulder_asymmetry"]),
        ("Trunk tilt",features.get("trunk_tilt_angle_abs", NaN),
         THRESHOLDS["trunk_tilt_angle_abs"]),
        ("Elbow diff",features.get("elbow_angle_difference", NaN),
         THRESHOLDS["elbow_angle_difference"]),
        ("Arm orient",features.get("upper_arm_orientation_difference", NaN),
         THRESHOLDS["upper_arm_orientation_difference"]),
        ("Wrist vert",features.get("normalized_wrist_vertical_difference", NaN),
         THRESHOLDS["normalized_wrist_vertical_difference"]),
        ("Wrist horiz",features.get("normalized_wrist_horizontal_difference", NaN),
         THRESHOLDS["normalized_wrist_horizontal_difference"]),
    ]

    panel_h = 30 + len(panel_items) * 24 + 30
    cv2.rectangle(frame, (5, 5), (320, panel_h), (0, 0, 0), -1)
    cv2.rectangle(frame, (5, 5), (320, panel_h), (80, 80, 80), 1)

    y_pos = 28
    for label, val, thresh in panel_items:
        if math.isnan(val):
            text  = f"{label}: N/A"
            color = (120, 120, 120)
            sym   = "--"
        elif val > thresh:
            text  = f"{label}: {val:.3f} > {thresh}"
            color = (0, 0, 255)
            sym   = "!!"
        else:
            text  = f"{label}: {val:.3f} <= {thresh}"
            color = (0, 200, 0)
            sym   = "OK"

        cv2.putText(frame, sym,  (10, y_pos),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 2)
        cv2.putText(frame, text, (38, y_pos),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.42, color, 1)
        y_pos += 24

    risk_color = {"low":    (0, 200, 0),
                  "medium": (0, 165, 255),
                  "high":   (0, 0, 255)}.get(rl, (150, 150, 150))
    cv2.rectangle(frame, (5, panel_h - 24), (320, panel_h), risk_color, -1)
    cv2.putText(frame, f"RISK: {rl.upper()} | {features.get('stroke','').upper()}",
                (10, panel_h - 8), cv2.FONT_HERSHEY_SIMPLEX,
                0.55, (255, 255, 255), 2)

    cv2.imwrite(output_path, frame)


# ══════════════════════════════════════════════════════════════════════
#  ⑨ PROCESS ONE JSON FILE
# ══════════════════════════════════════════════════════════════════════
def process_json(json_data: dict, stroke: str,
                 all_rows: list, frame_counter: list) -> dict:
    """
    Process all videos in one JSON file.
    Appends rows to all_rows list in place.
    Returns summary stats per video.
    """
    summary = {}

    for video_name, video_data in json_data.items():
        # Skip if wrong stroke or error
        if video_data.get("stroke", stroke) != stroke:
            continue
        if "error" in video_data:
            print(f"  ⚠️  Skipping {video_name} — {video_data['error']}")
            continue

        frames_iter = get_frames_iter(video_data)
        if not frames_iter:
            print(f"  ⚠️  No frames in {video_name}")
            continue

        print(f"\n  {'─'*50}")
        print(f"  Video  : {video_name}  ({stroke})")
        print(f"  Frames : {len(frames_iter)}")

        vid_annot_dir = Path(ANNOTATED_DIR) / video_name
        vid_annot_dir.mkdir(parents=True, exist_ok=True)

        vid_valid   = 0
        vid_invalid = 0
        vid_flags   = {"low": 0, "medium": 0, "high": 0}

        for frame_name, frame_data in tqdm(frames_iter,
                                           desc=f"  {video_name}",
                                           leave=False):
            if frame_data is None:
                continue

            keypoints = frame_data.get("keypoints", {})
            valid_kp, quality, n_valid = validate_keypoints(keypoints)

            if quality == "invalid":
                vid_invalid += 1
                continue

            vid_valid += 1

            features = calculate_features(
                valid_kp,
                str(frame_counter[0]),
                str(frame_name),
                quality,
                video_name,
                stroke
            )
            frame_counter[0] += 1

            flags, n_flags, risk_level = generate_risk_flags(features)
            features["risk_flags"]       = " | ".join(flags) if flags else "none"
            features["number_of_flags"]  = n_flags
            features["frame_risk_level"] = risk_level
            vid_flags[risk_level] += 1

            all_rows.append(features)

            # Visualization
            saved_path = frame_data.get("frame_path", "")
            frame_path = find_frame_on_disk(
                video_name, str(frame_name), saved_path, stroke
            )
            if frame_path:
                out_path = str(vid_annot_dir / f"annot_{Path(str(frame_name)).name}")
                draw_on_frame(frame_path, valid_kp, features, out_path)

        total = vid_valid + vid_invalid
        print(f"  Processed : {vid_valid}/{total} valid frames")
        print(f"  Risk      : low={vid_flags['low']} | "
              f"medium={vid_flags['medium']} | high={vid_flags['high']}")

        summary[video_name] = {
            "valid":   vid_valid,
            "invalid": vid_invalid,
            "flags":   vid_flags,
        }

    return summary


# ══════════════════════════════════════════════════════════════════════
#  ⑩ MAIN
# ══════════════════════════════════════════════════════════════════════
def run_angle_calculation():
    print("\n" + "═"*55)
    print("  PHASE 1: BIOMECHANICAL ANGLE CALCULATION")
    print("  Strokes: Breaststroke + Butterfly")
    print(f"  Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("═"*55)

    Path(ANNOTATED_DIR).mkdir(parents=True, exist_ok=True)

    all_rows      = []
    frame_counter = [0]  # mutable so it can be updated inside function

    # ── Breaststroke ─────────────────────────────────────────────────
    print("\n📂 Loading breaststroke JSON...")
    breast_data = load_json(BREASTSTROKE_JSON)
    if breast_data:
        print(f"  Found {len(breast_data)} breaststroke videos")
        process_json(breast_data, "breaststroke", all_rows, frame_counter)

    # ── Butterfly ────────────────────────────────────────────────────
    print("\n📂 Loading butterfly JSON...")
    butter_data = load_json(BUTTERFLY_JSON)
    if butter_data:
        print(f"  Found {len(butter_data)} butterfly videos")
        process_json(butter_data, "butterfly", all_rows, frame_counter)

    # ── Save CSV ──────────────────────────────────────────────────────
    if not all_rows:
        print("\n❌ No features extracted")
        return None

    df = pd.DataFrame(all_rows)
    df.to_csv(CSV_SAVE_PATH, index=False)

    # ── Summary ───────────────────────────────────────────────────────
    print("\n" + "═"*55)
    print("  PHASE 1 COMPLETE")
    print("═"*55)
    print(f"  Total frames processed : {len(df)}")
    print(f"  Features per frame     : {len(df.columns)}")
    print(f"  CSV saved → {CSV_SAVE_PATH}")
    print(f"  Annotated frames → {ANNOTATED_DIR}")

    # Per stroke breakdown
    for stroke in ["breaststroke", "butterfly"]:
        sdf = df[df["stroke"] == stroke]
        if len(sdf) == 0:
            continue
        print(f"\n  {stroke.upper()} ({len(sdf)} frames):")
        print(f"    Risk: low={len(sdf[sdf.frame_risk_level=='low'])} | "
              f"medium={len(sdf[sdf.frame_risk_level=='medium'])} | "
              f"high={len(sdf[sdf.frame_risk_level=='high'])}")

        for col in ["normalized_shoulder_asymmetry",
                    "trunk_tilt_angle_abs",
                    "elbow_angle_difference",
                    "arm_symmetry_score"]:
            if col in sdf.columns:
                valid = sdf[col].dropna()
                if len(valid) > 0:
                    print(f"    {col:<42}: "
                          f"{valid.mean():.3f} ± {valid.std():.3f}")

    print("\n  ✅ Ready for Phase 2 (injury_detection_fl.py)")
    print("═"*55)

    return df


# ══════════════════════════════════════════════════════════════════════
#  ⑪ RUN
# ══════════════════════════════════════════════════════════════════════
df_features = run_angle_calculation()

✅ Libraries loaded

═══════════════════════════════════════════════════════
  PHASE 1: BIOMECHANICAL ANGLE CALCULATION
  Strokes: Breaststroke + Butterfly
  Started: 2026-06-06 00:52:53
═══════════════════════════════════════════════════════

📂 Loading breaststroke JSON...
  Found 4 breaststroke videos

  ──────────────────────────────────────────────────
  Video  : breaststroke_front_S05  (breaststroke)
  Frames : 28


  Processed : 28/28 valid frames
  Risk      : low=0 | medium=28 | high=0

  ──────────────────────────────────────────────────
  Video  : breaststroke_front_S06  (breaststroke)
  Frames : 22


  Processed : 22/22 valid frames
  Risk      : low=3 | medium=18 | high=1

  ──────────────────────────────────────────────────
  Video  : breaststroke_front_S07  (breaststroke)
  Frames : 30


  Processed : 28/30 valid frames
  Risk      : low=6 | medium=22 | high=0

  ──────────────────────────────────────────────────
  Video  : breaststroke_front_S08  (breaststroke)
  Frames : 17


  Processed : 17/17 valid frames
  Risk      : low=6 | medium=11 | high=0

📂 Loading butterfly JSON...
  Found 3 butterfly videos

  ──────────────────────────────────────────────────
  Video  : butterfly_front_S07  (butterfly)
  Frames : 162


  Processed : 54/162 valid frames
  Risk      : low=0 | medium=47 | high=7

  ──────────────────────────────────────────────────
  Video  : butterfly_front_S08  (butterfly)
  Frames : 179


  Processed : 76/179 valid frames
  Risk      : low=0 | medium=60 | high=16

  ──────────────────────────────────────────────────
  Video  : butterfly_front_S09  (butterfly)
  Frames : 136


  Processed : 54/136 valid frames
  Risk      : low=0 | medium=33 | high=21

═══════════════════════════════════════════════════════
  PHASE 1 COMPLETE
═══════════════════════════════════════════════════════
  Total frames processed : 279
  Features per frame     : 53
  CSV saved → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/biomechanical_features_new.csv
  Annotated frames → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/annotated_frames_new

  BREASTSTROKE (95 frames):
    Risk: low=15 | medium=79 | high=1
    normalized_shoulder_asymmetry             : 0.022 ± 0.027
    trunk_tilt_angle_abs                      : 5.814 ± 4.018
    elbow_angle_difference                    : 10.004 ± 13.952
    arm_symmetry_score                        : 0.436 ± 0.236

  BUTTERFLY (184 frames):
    Risk: low=0 | medium=140 | high=44
    normalized_shoulder_asymmetry             : 0.025 ± 0.028
    trunk_tilt_angle_abs                      : 24.847 ± 23.314
    elbow_a